# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [ ]:
# Uncomment for Colab:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

## 1. Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib

# Detect environment and find Project Root
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # In Colab: search for project folder in Drive
    possible_roots = [
        Path('/content/drive/MyDrive/Final_Project_Deep_Learning'),
        Path('/content/drive/MyDrive/Final_Project_Deep_Lea'),
        Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')
    ]
    PROJECT_ROOT = None
    for path in possible_roots:
        if path.exists() and (path / 'local_main.ipynb').exists():
            PROJECT_ROOT = path
            break
    
    if PROJECT_ROOT is None:
        print("❌ Error: Could not find 'Final_Project_Deep_Learning' in Drive.")
        print("   Please verify the folder name and that you mounted Drive.")
        PROJECT_ROOT = Path.cwd()
    else:
        print(f"✅ Found Project Root: {PROJECT_ROOT}")
else:
    # Local: use current working directory
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'local_main.ipynb').exists():
        for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (p / 'local_main.ipynb').exists():
                PROJECT_ROOT = p
                break

os.chdir(PROJECT_ROOT)

# Define data and checkpoint directories
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Add to sys.path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import project modules
import models.utils as utils
importlib.reload(utils)
from models import model_A as ma

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nConfiguration Complete:")
print(f"   - Device: {device}")
print(f"   - Working Directory: {os.getcwd()}")
print(f"   - Data Directory: {DATA_DIR}")
print(f"   - Checkpoints Directory: {CHECKPOINT_DIR}")

## 2. Data Loading and Preprocessing

**Two modes of operation:**

1. **Training Mode** (`DOWNLOAD_DATA = True`): 
   - Downloads MUSDB18 dataset (~4GB, 144 tracks)
   - Prepares training data with realistic mixture weights
   - Required for training from scratch
   
2. **Inference-Only Mode** (`DOWNLOAD_DATA = False`):
   - Skip data download (default)
   - Use existing model checkpoints for demonstrations
   - Upload your own songs for separation
   - Perfect for quick demos and evaluation

**Mixture Weights** (for those who download):
- Vocals: 35%, Drums: 30%, Bass: 20%, Other: 15%
- Matches real music production standards

In [ ]:
DOWNLOAD_DATA = False  # Change to True to download MUSDB18 dataset (~4GB)
FORCE_REBUILD = False  # Set True to regenerate data even if it exists

if DOWNLOAD_DATA:
    import musdb
    print("Downloading MUSDB18 dataset (~4GB)...")
    mus = musdb.DB(download=True)
    utils.prepare_curriculum_cache(mus=mus, cache_dir=str(DATA_DIR), sr=22050, force_rebuild=FORCE_REBUILD)
    print(f"✅ Dataset ready: {len(mus.tracks)} tracks")

# Load file lists if data exists
try:
    mix_files_stage1, tgt_files_stage1, mix_files_stage2, tgt_files_stage2 = utils.get_curriculum_file_lists(cache_dir=str(DATA_DIR))
    print(f"✅ Training data found: Stage 1 ({len(mix_files_stage1)}), Stage 2 ({len(mix_files_stage2)})")
except Exception as e:
    print("⚠️  No training data found (OK for inference-only mode)")
    print(f"   Details: {e}")
    mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

## 3. Overfit Sanity Check

Test model's ability to overfit on a single song (validates implementation).

In [ ]:
import importlib
import gc
importlib.reload(utils)

# ============================================================================
# AUTOMATED PRESET COMPARISON (Memory-Safe + Tensor-Safe)
# ============================================================================
# Testing deeper networks: 4, 5, and 6 layers with appropriately scaled chunks

PRESETS = [
    {
        'name': 'Small (4.0s, 4 layers, 128 filters)',
        'chunk_duration': 4.0,
        'num_layers': 4,
        'base_filters': 128,
    },
    {
        'name': 'Medium (6.0s, 4 layers, 256 filters)',
        'chunk_duration': 6.0,
        'num_layers': 4,
        'base_filters': 256,
    },
    {
        'name': 'Large (8.0s, 4 layers, 256 filters)',
        'chunk_duration': 8.0,
        'num_layers': 4,
        'base_filters': 256,
    },
    {
        'name': 'Deeper-5 (10.0s, 5 layers, 128 filters)',
        'chunk_duration': 10.0,
        'num_layers': 5,
        'base_filters': 128,
    },
    {
        'name': 'Deepest-6 (14.0s, 6 layers, 128 filters)',
        'chunk_duration': 14.0,
        'num_layers': 6,
        'base_filters': 128,
    },
]

# START FROM PRESET 3 (LARGE) - SKIP 1 & 2
PRESETS_TO_TEST = PRESETS[2:]  # Start from Large (index 2)

print(f"\n{'='*70}")
print(f"AUTOMATED PRESET COMPARISON - Testing {len(PRESETS_TO_TEST)} architectures")
print(f"(Starting from Large (4-layer) → 5 layers → 6 layers)")
print(f"(Presets 1-2 already trained, using existing checkpoints)")
print(f"{'='*70}\n")

results = []

# Load existing results from presets 1-2
print("Loading existing results from presets 1-2...\n")
for idx_existing in [1, 2]:
    existing_ckpt = CHECKPOINT_DIR / f"debug_overfit_{idx_existing}_preset.pth"
    if existing_ckpt.exists():
        try:
            print(f"✅ Loading preset {idx_existing}: {PRESETS[idx_existing-1]['name']}")
            ckpt_data = torch.load(existing_ckpt, map_location=device)
            
            # Extract available data from checkpoint
            if 'history' in ckpt_data:
                history = ckpt_data['history']
                final_train_loss = history['train_loss'][-1] if 'train_loss' in history else float('inf')
                final_val_loss = history['val_loss'][-1] if 'val_loss' in history else float('inf')
                min_val_loss = min(history['val_loss']) if 'val_loss' in history else float('inf')
                num_epochs = len(history['train_loss']) if 'train_loss' in history else 0
            else:
                final_train_loss = final_val_loss = min_val_loss = float('inf')
                num_epochs = 0
                history = {}
            
            # Get param count from model config
            total_params = sum(p.numel() for p in ma.TimeFrequencyDomainUNet(
                in_channels=1, out_channels=1,
                base_filters=PRESETS[idx_existing-1]['base_filters'],
                num_layers=PRESETS[idx_existing-1]['num_layers']
            ).parameters())
            
            results.append({
                'name': PRESETS[idx_existing-1]['name'],
                'final_train_loss': final_train_loss,
                'final_val_loss': final_val_loss,
                'min_val_loss': min_val_loss,
                'num_epochs': num_epochs,
                'params': total_params,
                'history': history,
                'ckpt_path': str(existing_ckpt)
            })
            print(f"   Train Loss: {final_train_loss:.6f} | Val Loss: {final_val_loss:.6f}\n")
        except Exception as e:
            print(f"⚠️  Could not load preset {idx_existing}: {str(e)[:50]}\n")

# Now test new presets (3+)
print("\nTesting new presets (3+)...\n")

for idx, preset in enumerate(PRESETS_TO_TEST, start=3):  # Start numbering from 3
    print(f"\n[{idx}/{len(PRESETS)}] Testing: {preset['name']}")
    print(f"-" * 70)
    
    # Configure
    OVERFIT_CONFIG = utils.get_overfit_config(
        chunk_duration=preset['chunk_duration'],
        num_layers=preset['num_layers']
    )
    BASE_FILTERS = preset['base_filters']
    MODEL_NAME = preset['name']
    
    print(f"Config: Chunk={OVERFIT_CONFIG['chunk_duration']}s | Layers={OVERFIT_CONFIG['num_layers']} | Filters={BASE_FILTERS}")
    
    # Build model
    overfit_device = device
    overfit_processor = utils.AudioProcessor(device=overfit_device)
    
    overfit_model = ma.TimeFrequencyDomainUNet(
        in_channels=1,
        out_channels=1,
        base_filters=BASE_FILTERS,
        num_layers=OVERFIT_CONFIG['num_layers'],
        batchnorm=True,
        dropout=0.0
    ).to(overfit_device)
    
    total_params = sum(p.numel() for p in overfit_model.parameters())
    print(f"Parameters: {total_params:,}")
    
    # Train
    overfit_loss_fn = nn.MSELoss()
    overfit_optimizer = optim.Adam(overfit_model.parameters(), lr=OVERFIT_CONFIG['learning_rate'])
    
    overfit_ckpt = CHECKPOINT_DIR / f"debug_overfit_{idx}_preset.pth"
    
    # Clean old checkpoint
    if overfit_ckpt.exists():
        overfit_ckpt.unlink()
    epochs_dir = CHECKPOINT_DIR / f"debug_overfit_{idx}_preset_epochs"
    if epochs_dir.exists():
        import shutil
        shutil.rmtree(epochs_dir)
    
    # Run training with error handling
    try:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
        
        # Extract metrics
        if history_overfit and 'train_loss' in history_overfit:
            final_train_loss = history_overfit['train_loss'][-1]
            final_val_loss = history_overfit['val_loss'][-1]
            min_val_loss = min(history_overfit['val_loss'])
            num_epochs = len(history_overfit['train_loss'])
        else:
            final_train_loss = float('inf')
            final_val_loss = float('inf')
            min_val_loss = float('inf')
            num_epochs = 0
        
        results.append({
            'name': MODEL_NAME,
            'final_train_loss': final_train_loss,
            'final_val_loss': final_val_loss,
            'min_val_loss': min_val_loss,
            'num_epochs': num_epochs,
            'params': total_params,
            'history': history_overfit,
            'ckpt_path': str(overfit_ckpt)
        })
        
        print(f"✅ Final Train Loss: {final_train_loss:.6f} | Final Val Loss: {final_val_loss:.6f} | Best Val: {min_val_loss:.6f}")
    
    except ZeroDivisionError as e:
        print(f"❌ ERROR: No valid chunks found for {preset['chunk_duration']}s duration")
        print(f"   The randomly selected song is shorter than {preset['chunk_duration']}s")
        print(f"   → SKIPPING this preset")
        print(f"   Note: Try reducing chunk_duration or use longer songs")
    except Exception as e:
        print(f"❌ ERROR during training: {type(e).__name__}: {str(e)[:100]}")
        print(f"   → SKIPPING this preset")
    finally:
        # ⚠️ MEMORY CLEANUP - CRITICAL FOR COLAB
        print(f"🧹 Cleaning up memory...")
        del overfit_model, overfit_optimizer, overfit_processor, overfit_loss_fn
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ Memory cleaned!\n")

# ============================================================================
# COMPARISON AND WINNER ANNOUNCEMENT
# ============================================================================
print(f"\n\n{'='*70}")
print("RESULTS SUMMARY - Full Layer Depth Comparison (Presets 1-5)")
print(f"{'='*70}\n")

if results:
    import pandas as pd
    df_results = pd.DataFrame([
        {
            'Architecture': r['name'],
            'Final Train Loss': f"{r['final_train_loss']:.6f}",
            'Final Val Loss': f"{r['final_val_loss']:.6f}",
            'Best Val Loss': f"{r['min_val_loss']:.6f}",
            'Epochs': r['num_epochs'],
            'Parameters': f"{r['params']:,}"
        }
        for r in results
    ])

    print(df_results.to_string(index=False))

    # Find winner by lowest final val loss
    best_result = min(results, key=lambda x: x['final_val_loss'])
    print(f"\n🏆 BEST PRESET: {best_result['name']}")
    print(f"   Final Val Loss: {best_result['final_val_loss']:.6f}")
    print(f"   Parameters: {best_result['params']:,}")

    print(f"\n✅ Comparison complete! Best config saved for reference.")
else:
    print("⚠️  No presets completed successfully.")
    print("   All presets either failed or were skipped.")
    print("   → Try reducing chunk_duration values")



### Run Overfit Training

Only runs if checkpoint doesn't exist. Set `SKIP_OVERFIT = True` to skip.

In [ ]:
### Compare All Presets Visually

# Plot all preset results on same graph
if results:
    plt.figure(figsize=(14, 6))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for idx, result in enumerate(results):
        if result['history'] and 'val_loss' in result['history']:
            epochs = range(1, len(result['history']['val_loss']) + 1)
            plt.plot(epochs, result['history']['val_loss'], 
                    label=result['name'], linewidth=2.5, marker='o', markersize=5, color=colors[idx])
    
    plt.xlabel("Epoch", fontsize=12, fontweight='bold')
    plt.ylabel("Validation Loss", fontsize=12, fontweight='bold')
    plt.title("Overfit Validation Loss Comparison: All Presets", fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Bar chart of final losses
    plt.figure(figsize=(12, 5))
    names = [r['name'].split('(')[0].strip() for r in results]
    final_losses = [r['final_val_loss'] for r in results]
    
    bars = plt.bar(names, final_losses, color=colors[:len(results)], alpha=0.7, edgecolor='black', linewidth=2)
    
    # Highlight best
    best_idx = final_losses.index(min(final_losses))
    bars[best_idx].set_color('gold')
    bars[best_idx].set_edgecolor('darkgreen')
    bars[best_idx].set_linewidth(3)
    
    plt.ylabel("Final Validation Loss", fontsize=12, fontweight='bold')
    plt.title("Final Loss Comparison (Winner in Gold)", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print("✅ Comparison plots generated!")


In [ ]:
SKIP_OVERFIT = False
FORCE_RETRAIN = True  # Set True to retrain even if checkpoint exists

if not SKIP_OVERFIT:
    overfit_ckpt = CHECKPOINT_DIR / "debug_overfit_1song.pth"
    
    # Clean up old checkpoint if retraining
    if FORCE_RETRAIN and overfit_ckpt.exists():
        overfit_ckpt.unlink()
        # Also clean up epochs folder
        epochs_dir = CHECKPOINT_DIR / "debug_overfit_1song_epochs"
        if epochs_dir.exists():
            import shutil
            shutil.rmtree(epochs_dir)
        print("🗑️  Deleted old checkpoint and epochs")
    
    print(f"\n{'='*60}")
    print("Overfit Sanity Check")
    print(f"{'='*60}")
    checkpoint_exists = utils.check_checkpoint(overfit_ckpt, "Overfit Checkpoint")
    
    if not checkpoint_exists and len(mix_files_stage1) > 0:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
    elif len(mix_files_stage1) == 0:
        print("\n⚠️  No training data available.")
        print("   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.")
        history_overfit = {}
    else:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
else:
    print("Skipping overfit test")
    history_overfit = {}

### View Overfit Results

In [ ]:
# Plot overfit learning curve from checkpoint or memory
overfit_ckpt_path = CHECKPOINT_DIR / "debug_overfit_1song.pth"

if overfit_ckpt_path.exists():
    utils.plot_loss_from_checkpoint(str(overfit_ckpt_path), title="Overfit Learning Curve (1 Song)")
elif history_overfit:
    utils.plot_loss_history(history_overfit, title="Overfit Learning Curve (1 Song)")
else:
    print("No overfit results available. Run the overfit training cell above.")

## 4. Model Architecture

View the U-Net model structure and parameter count.

In [ ]:
# Model A architecture summary
model_summary = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=64,
    num_layers=4
).to(device)
print(model_summary)
del model_summary

## 5. Model Configuration

Configure model architecture and training hyperparameters.

In [ ]:
# Get configurations from utils
MODEL_CONFIG = utils.get_model_a_config()
TRAIN_CONFIG = utils.get_training_config()

processor = utils.AudioProcessor(device=device)

model = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=64,
    num_layers=4,
    batchnorm=True,
    dropout=0.1
).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=TRAIN_CONFIG['learning_rate'])

print("Model A setup complete.")

## 6. Full Training (Curriculum Learning)

Train the model on the full MUSDB18 dataset in two stages.

In [ ]:
SKIP_TRAINING = False  # Set to True to skip full training

if not SKIP_TRAINING:
    # Check if checkpoints exist
    ckpt_s1 = CHECKPOINT_DIR / "full_stage1.pth"
    ckpt_s2 = CHECKPOINT_DIR / "full_stage2.pth"
    print(f"\n{'='*60}")
    print("Full Training: Curriculum Learning")
    print(f"{'='*60}")
    
    s1_exists = utils.check_checkpoint(ckpt_s1, "Stage 1 Checkpoint")
    s2_exists = utils.check_checkpoint(ckpt_s2, "Stage 2 Checkpoint")
    
    # Only train if data exists and checkpoints don't
    if len(mix_files_stage1) > 0 and len(mix_files_stage2) > 0:
        if not s1_exists or not s2_exists:
            print("\nStarting Full Training Pipeline...")
            hist_s1, hist_s2 = utils.run_full_training(
                model=model,
                processor=processor,
                optimizer=optimizer,
                loss_fn=loss_fn,
                train_config=TRAIN_CONFIG,
                cache_dir=str(DATA_DIR),
                save_path_stage1=str(ckpt_s1),
                save_path_stage2=str(ckpt_s2),
                device=device,
            )
        else:
            print("\n✅ Both checkpoints exist. Training skipped.")
            hist_s1, hist_s2 = {}, {}
    else:
        print("\n⚠️  No training data available. Skipping full training.")
        print("   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.")
        hist_s1, hist_s2 = {}, {}
else:
    print("Skipping full training (SKIP_TRAINING = True)")
    hist_s1, hist_s2 = {}, {}

## 7. View Training Results

Plot loss curves from saved checkpoint.

In [ ]:
# Plot training curves - reads from epoch folders if available!
import re
import matplotlib.pyplot as plt
from pathlib import Path

stage2_ckpt = CHECKPOINT_DIR / "full_stage2.pth"
stage2_epochs_dir = CHECKPOINT_DIR / "full_stage2_epochs"

# Try epoch folder first (has complete history)
if stage2_epochs_dir.exists():
    print(f"📁 Reading from: {stage2_epochs_dir.name}")
    
    epoch_files = sorted(stage2_epochs_dir.glob("epoch_*.txt"))
    train_losses = []
    val_losses = []
    
    for epoch_file in epoch_files:
        with open(epoch_file, 'r') as f:
            content = f.read()
            train_match = re.search(r'Train Loss = ([\d.]+)', content)
            val_match = re.search(r'Val Loss = ([\d.]+)', content)
            if train_match and val_match:
                train_losses.append(float(train_match.group(1)))
                val_losses.append(float(val_match.group(1)))
    
    if train_losses:
        print(f"✅ Found {len(train_losses)} epochs\n")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        epochs = range(1, len(train_losses) + 1)
        ax.plot(epochs, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=6)
        ax.plot(epochs, val_losses, 's--', label='Val Loss', linewidth=2, markersize=6)
        ax.set_title("Full Training: Stage 2", fontsize=14, fontweight='bold')
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        print(f"📊 Stats: Epochs={len(train_losses)} | Best Val={min(val_losses):.6f} (epoch {val_losses.index(min(val_losses))+1})")
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Epoch files found but couldn't parse them")
        
elif stage2_ckpt.exists():
    utils.plot_loss_from_checkpoint(str(stage2_ckpt), title="Full Training: Stage 2")
elif hist_s2:
    utils.plot_loss_history(hist_s2, title="Full Training: Stage 2")
else:
    print("No training results available. Run the full training cell above.")

## 8. Listen to Separated Audio

Demo the model on MUSDB18 samples with audio playback and spectrograms.

In [ ]:
# Preview a sample separation from the cache
utils.demo_separation_sample(
    model=model,
    processor=processor,
    cache_dir=str(DATA_DIR),
    stage="stage1",
    song_num=12,
    duration=6,
    sr=22050,
    device=device,
    play_audio_output=True,
 )

In [ ]:
import os, sys
print('Current working directory:', os.getcwd())
print('sys.path:', sys.path)
print('Directory listing:', os.listdir('.'))

# Model B2: [Architecture Name]

[Description of Model B2]

# Model B1: [Architecture Name]

[Description of Model B1]